# Multilayer Perceptron

In [10]:
from analyses.population_analysis import get_all_combined_exploded_spike_counts
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

spike_counts = get_all_combined_exploded_spike_counts(group_name='Zombies', apply_filter=True, apply_binning=True, bin_size=0.2)
spike_counts['NeuronID'].nunique()

322

In [3]:
import numpy as np
spike_counts_short = spike_counts[spike_counts['TimeBinIndex'] < 15].copy()
# Step 1: Filter to only neurons with exactly 40 bins for *each* TaskField
# --------------------------------------------------------------
# Count number of time bins per (TaskField, NeuronID)
bin_counts = (
    spike_counts_short
    .groupby(['TaskField', 'NeuronID'])['TimeBinIndex']
    .nunique()
    .reset_index(name='BinCount')
)

# Keep only those (TaskField, NeuronID) pairs with exactly 40 bins
valid_pairs = bin_counts[bin_counts['BinCount'] == 15][['TaskField', 'NeuronID']]

In [4]:

# Inner merge to keep only valid spike counts
filtered_df = spike_counts.merge(valid_pairs, on=['TaskField', 'NeuronID'])

# Step 2: Pivot to create wide format (TaskField × [NeuronID × TimeBinIndex])
# --------------------------------------------------------------
# Create unique feature name per neuron × time bin
filtered_df['Feature'] = filtered_df['NeuronID'] + '_bin' + filtered_df['TimeBinIndex'].astype(str)

# Pivot: one row per TaskField, one column per Neuron×Bin
pivot_df = (
    filtered_df
    .pivot_table(index='TaskField', columns='Feature', values='SpikeCount', fill_value=0)
)

# Step 3: Get labels (MonkeyName) for each TaskField
# --------------------------------------------------------------
# Create label vector: assume all rows with same TaskField have the same MonkeyName
task_to_label = (
    spike_counts
    .drop_duplicates(subset=['TaskField'])[['TaskField', 'MonkeyName']]
    .set_index('TaskField')
)

# Align labels to pivot_df
y = task_to_label.loc[pivot_df.index, 'MonkeyName'].values

# Step 4: Final X matrix
# --------------------------------------------------------------
X = pivot_df.values
feature_names = pivot_df.columns.tolist()

# Optional: print shape
print("X shape:", X.shape)
print("y shape:", y.shape)



X shape: (82, 90)
y shape: (82,)


In [5]:
spike_counts_single_unit = spike_counts[spike_counts['NeuronID'].str.contains('Unit')]
spike_counts_single_unit['NeuronID'].nunique()

65

In [6]:
# Step 0: Restrict to bins 200–600ms (bins 4 to 8)

spike_counts_window = spike_counts_single_unit[
    (spike_counts_single_unit['TimeBinIndex'] >= 1) &
    (spike_counts_single_unit['TimeBinIndex'] <= 5)
].copy()

# Step 1: Keep only neurons with full coverage of 8 bins in this window
bin_counts = (
    spike_counts_window
    .groupby(['TaskField', 'NeuronID'])['TimeBinIndex']
    .nunique()
    .reset_index(name='BinCount')
)

valid_pairs = bin_counts[bin_counts['BinCount'] == 5][['TaskField', 'NeuronID']]
filtered_df = spike_counts_window.merge(valid_pairs, on=['TaskField', 'NeuronID'])

# Step 2: Create feature names like NeuronID_bin5
filtered_df['Feature'] = (
    filtered_df['NeuronID'] + '_bin' + filtered_df['TimeBinIndex'].astype(str)
)

# Step 3: Pivot
pivot_df = filtered_df.pivot_table(
    index='TaskField',
    columns='Feature',
    values='SpikeCount',
    fill_value=0
)


# Step 4: Get MonkeyName labels
task_to_label = (
    spike_counts.drop_duplicates(subset=['TaskField'])[['TaskField', 'MonkeyName']]
    .set_index('TaskField')
)

y = task_to_label.loc[pivot_df.index, 'MonkeyName'].values
X = pivot_df.values
feature_names = pivot_df.columns.tolist()

# Optional: check shape
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1149, 325)
y shape: (1149,)


In [9]:
# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 2: Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
from sklearn.decomposition import PCA
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Step 3: Train the MLPClassifier
mlp = MLPClassifier(
    hidden_layer_sizes=(1000,),
    activation='relu',
    solver='sgd',
    alpha=0.01,
    max_iter=10000
)
mlp.fit(X_train_pca, y_train)

# Step 4: Evaluate
y_pred = mlp.predict(X_test_pca)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

        110E       0.00      0.00      0.00        25
        143H       0.09      0.08      0.08        26
        151J       0.08      0.08      0.08        25
         67G       0.11      0.12      0.11        25
         69X       0.14      0.19      0.16        26
        7124       0.12      0.12      0.12        26
         72X       0.21      0.15      0.18        26
         87J       0.15      0.19      0.17        26
         94B       0.08      0.08      0.08        25

    accuracy                           0.11       230
   macro avg       0.11      0.11      0.11       230
weighted avg       0.11      0.11      0.11       230

Confusion Matrix:
[[0 4 3 4 4 1 1 3 5]
 [1 2 1 4 8 5 1 2 2]
 [2 0 2 4 5 1 2 7 2]
 [4 2 2 3 1 5 3 2 3]
 [1 2 1 3 5 4 2 4 4]
 [2 3 5 4 4 3 1 3 1]
 [2 2 3 1 2 4 4 3 5]
 [3 4 4 2 4 1 1 5 2]
 [1 3 5 3 2 1 4 4 2]]
